# 컬러 이펙트 LoRA 3바퀴 — 프롬프트·강도·부정어 조합 시험 (Kaggle T4, 재학습 없음)
2차 결과에서 폭발·불이 회청색 연기 위주로 나왔다(2026-10-26). 원인 후보 중 생성 쪽만 먼저 본다.
- 프롬프트 3종: A 원래 폭발(smoke·billowing mass 포함) / B 폭발에서 smoke 뺌 + vivid colors / C 불에서 smoke 뺌 + vivid colors
- LoRA 강도 0.6 · 0.8 · 1.0 (+ LoRA 없는 기본 모델 줄)
- 부정어 2종: 기본 / + monochrome, greyscale, grey smoke
- 시드 2개(1234, 777). 4종(기본·0.6·0.8·1.0) × 12장 = 48장. 2회차(10-26): 강도를 alpha 배율 사본으로 적용(1회차는 합치기 모드가 강도를 무시), 약 30–40분.
- 설정: **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 끝나면 Output의 `promptgrid.jpg`(한 장)만 받아 채팅에 올리면 된다. 그림 전부는 `promptgrid_result.zip`.

In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 기본 모델(Hugging Face)과 2차 결과 zip(Drive 공유 링크)에서 컬러 LoRA 3바퀴만
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/v2.zip "https://drive.usercontent.google.com/download?id=1VjfgquMobDcMNxbwqc-0akpbcQ9BycqN&export=download&confirm=t"
!cd /tmp && unzip -q -o /tmp/v2.zip out/nkmr_fx_color-000003.safetensors && rm /tmp/v2.zip
!ls -la /tmp/ilxl.safetensors /tmp/out/

In [ ]:
# 3) 프롬프트 파일(부정어는 줄마다 --n), 강도별로 그림
import os, glob
P = {
 "A": "nkmr_fx, anime screencap, explosion, smoke, billowing mass, white core orange rim, layered cel shading, no outline, sharp tips, expanding, large effect, no humans, simple background, wide shot",
 "B": "nkmr_fx, vivid colors, anime screencap, explosion, fire, radial, white core orange rim, layered cel shading, no outline, sharp tips, peak, large effect, no humans, simple background, wide shot",
 "C": "nkmr_fx, vivid colors, anime screencap, fire, flames, pillar, white core orange rim, layered cel shading, sharp tips, expanding, large effect, no humans, city, night, from below",
}
NEG = {"n0": "lowres, worst quality, text, watermark",
       "n1": "lowres, worst quality, text, watermark, monochrome, greyscale, grey smoke, muted colors"}
SEEDS = [1234, 777]
lines, order = [], []
for nk, nv in NEG.items():
    for pk, pv in P.items():
        for sd in SEEDS:
            lines.append(f"{pv} --n {nv} --w 1344 --h 768 --s 28 --l 6 --d {sd}")
            order.append((nk, pk, sd))
open('/tmp/pg.txt', 'w').write("\n".join(lines))
L = "/tmp/out/nkmr_fx_color-000003.safetensors"
# --network_merge(LyCORIS 합치기)는 --network_mul을 무시했다(2026-10-26 1회차: 0.6·0.8·1.0 그림이 픽셀까지 같음).
# 강도 m은 LoCon의 alpha(효과 크기 = alpha/dim 비례)를 m배 한 사본으로 준다
from safetensors.torch import load_file, save_file
sd0 = load_file(L)
def scaled(m):
    f = f"/tmp/out/color3_x{m}.safetensors"
    if not os.path.exists(f):
        save_file({k: (v * m if k.endswith(".alpha") else v) for k, v in sd0.items()}, f)
    return f
runs = [("base", None), ("s0.6", 0.6), ("s0.8", 0.8), ("s1.0", 1.0)]
for name, mul in runs:
    out = f"/kaggle/working/pg/{name}"
    os.makedirs(out, exist_ok=True)
    net = f"--network_module lycoris.kohya --network_weights {scaled(mul)} --network_mul 1.0 --network_merge" if mul else ""
    os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/pg.txt --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a {net} > {out}/log.txt 2>&1")
    print(name, len(glob.glob(out + '/*.png')), 'images')

In [ ]:
# 4) 한 장 격자: 줄 = 강도 × 부정어(8줄), 칸 = 글 A·B·C × 시드 2(6칸)
from PIL import Image, ImageDraw
W0, H0 = 448, 256
G = Image.new('RGB', (W0 * 6, H0 * 8), 'black')
d = ImageDraw.Draw(G)
for ri, (name, _) in enumerate(runs):
    ims = sorted(glob.glob(f'/kaggle/working/pg/{name}/*.png'))
    for k, p in enumerate(ims[:24]):
        nk, pk, sd = order[k]
        row = ri * 2 + (0 if nk == "n0" else 1)
        col = "ABC".index(pk) * 2 + SEEDS.index(sd)
        G.paste(Image.open(p).resize((W0, H0)), (W0 * col, H0 * row))
        if col == 0:
            d.rectangle((0, H0 * row, 150, H0 * row + 20), fill='black'); d.text((4, H0 * row + 4), f"{name} {nk}", fill='white')
        if row == 0:
            d.rectangle((W0 * col, 0, W0 * col + 90, 20), fill='black'); d.text((W0 * col + 100 if col == 0 else W0 * col + 4, 4), f"{pk} seed{sd}", fill='white')
G.save('/kaggle/working/promptgrid.jpg', quality=88)
!cd /kaggle/working && zip -q -r promptgrid_result.zip pg promptgrid.jpg && ls -la /kaggle/working